# Pipeline: Counseling Count/Intensity Effect on Reintegration Readiness

## 1. Problem Framing
- **Business question:** Does higher counseling count/intensity in early case management improve future reintegration readiness?
- **Primary use:** prioritize counseling resources and identify residents likely to benefit from higher-intensity support.
- **Approach:** mixed explanatory + predictive.
  - Explanatory: estimate association and treatment-style effect of high counseling intensity.
  - Predictive: forecast 365-day readiness to support operational planning.
- **Outcome definition:** resident reaches readiness proxy (`target_ready_365d = 1`) if case closes in the future window after feature period.

## 2. Data Acquisition, Preparation, and Exploration

Data sources and joins:
- Tables used: `residents`, `process_recordings`, and `intervention_plans` from the Lighthouse CSV data folder.
- Join key: `resident_id` across all tables, with one resident-level modeling row after aggregation.
- Join logic: all counseling/plan features are restricted to the first 90 days after enrollment before joining to the target.

Leakage-safe framing:
- Feature window: first 90 days after enrollment.
- Target window: next 365 days after feature window ends.
- No outcome-period variables are used in features.

Data preparation and feature engineering (reproducible pipeline):
- Shared cleaning functions are applied first (`basic_cleaning`, summary helpers) to standardize data types and dates.
- Engineered features include counseling intensity and quality proxies: `counseling_count_90d`, `counseling_minutes_90d`, `avg_session_minutes_90d`, `progress_noted_rate_90d`, `individual_share_90d`.
- Plan adherence/context features include `plans_created_90d` and `plans_achieved_share_90d`.
- Age is parsed to numeric and missing values are handled consistently in code (not manual edits), making reruns reproducible.

Exploration conclusions from this run:
- Modeling frame size is 60 residents.
- Target prevalence is imbalanced (~26.7% ready, ~73.3% not ready), so recall/PR-aware metrics are important.
- Missingness after preparation is effectively resolved in the modeling frame.
- High-intensity treatment prevalence is ~30%, which is sufficient for comparison but still relatively small for stable causal-style estimates.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

# Reference shared cleaning/EDA helpers first.
sys.path.append(str(Path('ml-pipelines').resolve()))
from functions import basic_cleaning, missing_value_summary, numeric_summary, categorical_summary

# Resolve data path robustly for both execution contexts:
# - repo root cwd: ./lighthouse_csv_v7
# - notebook cwd (ml-pipelines): ../lighthouse_csv_v7
# - local backend snapshot: ./backend/csv or ../backend/csv
DATA_CANDIDATES = [
    Path.cwd() / 'lighthouse_csv_v7',
    Path.cwd().parent / 'lighthouse_csv_v7',
    Path.cwd() / 'backend' / 'csv',
    Path.cwd().parent / 'backend' / 'csv',
]
DATA = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "Missing data folder. Expected one of './lighthouse_csv_v7', '../lighthouse_csv_v7', './backend/csv', or '../backend/csv' from current notebook working directory."
    )
print(f"Using data folder: {DATA.resolve()}")

res = pd.read_csv(DATA / 'residents.csv')
proc = pd.read_csv(DATA / 'process_recordings.csv')
plans = pd.read_csv(DATA / 'intervention_plans.csv')

# Shared reusable cleaning first (required by project convention).
res = basic_cleaning(res, date_columns=['date_enrolled', 'date_closed'])
proc = basic_cleaning(proc, date_columns=['session_date'])
plans = basic_cleaning(plans, date_columns=['target_date', 'case_conference_date', 'created_at', 'updated_at'])

print('residents:', res.shape)
print('process_recordings:', proc.shape)
print('intervention_plans:', plans.shape)

Using data folder: C:\Users\joshu\OneDrive\IS CORE\realINTEXpt2\Intex-2\backend\csv
residents: (60, 49)
process_recordings: (2819, 15)
intervention_plans: (180, 11)


In [2]:
# Build leakage-safe panel anchored on enrollment.
base = res[[
    'resident_id', 'date_enrolled', 'date_closed', 'present_age',
    'case_category', 'reintegration_type', 'initial_risk_level'
]].dropna(subset=['date_enrolled']).copy()

base['feat_end'] = base['date_enrolled'] + pd.Timedelta(days=90)
base['target_end'] = base['feat_end'] + pd.Timedelta(days=365)
base['target_ready_365d'] = (
    (base['date_closed'].notna())
    & (base['date_closed'] > base['feat_end'])
    & (base['date_closed'] <= base['target_end'])
).astype(int)

# Counseling count/intensity features from process_recordings in feature window.
proc_m = base[['resident_id', 'date_enrolled', 'feat_end']].merge(proc, on='resident_id', how='left')
proc_m = proc_m[
    (proc_m['session_date'] >= proc_m['date_enrolled'])
    & (proc_m['session_date'] <= proc_m['feat_end'])
].copy()

proc_m['progress_noted_num'] = proc_m['progress_noted'].astype(str).str.lower().map({'true': 1, 'false': 0})
proc_m['session_type'] = proc_m['session_type'].fillna('Unknown')

proc_agg = proc_m.groupby('resident_id').agg(
    counseling_count_90d=('recording_id', 'count'),
    counseling_minutes_90d=('session_duration_minutes', 'sum'),
    avg_session_minutes_90d=('session_duration_minutes', 'mean'),
    progress_noted_rate_90d=('progress_noted_num', 'mean'),
    individual_share_90d=('session_type', lambda s: (s == 'Individual').mean()),
).reset_index()

# Lightweight plan adherence features in feature window.
plan_m = base[['resident_id', 'date_enrolled', 'feat_end']].merge(plans, on='resident_id', how='left')
plan_m = plan_m[
    (plan_m['created_at'] >= plan_m['date_enrolled'])
    & (plan_m['created_at'] <= plan_m['feat_end'])
].copy()

plan_agg = plan_m.groupby('resident_id').agg(
    plans_created_90d=('plan_id', 'count'),
    plans_achieved_share_90d=('status', lambda s: (s == 'Achieved').mean()),
).reset_index()

df = (
    base[['resident_id', 'date_enrolled', 'target_ready_365d', 'present_age', 'case_category', 'reintegration_type', 'initial_risk_level']]
    .merge(proc_agg, on='resident_id', how='left')
    .merge(plan_agg, on='resident_id', how='left')
)

# Fix age parsing: values are strings like "17 Years 6 months".
df['present_age'] = (
    df['present_age'].astype(str).str.extract(r'(\d+(?:\.\d+)?)')[0]
)

num_cols_all = [
    'present_age', 'counseling_count_90d', 'counseling_minutes_90d',
    'avg_session_minutes_90d', 'progress_noted_rate_90d', 'individual_share_90d',
    'plans_created_90d', 'plans_achieved_share_90d'
]
for c in num_cols_all:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

for c in ['case_category', 'reintegration_type', 'initial_risk_level']:
    df[c] = df[c].fillna('Unknown').astype(str)

# Define treatment with stricter intensity threshold (upper tercile) to avoid weak median split.
count_cut = df['counseling_count_90d'].quantile(0.67)
minute_cut = df['counseling_minutes_90d'].quantile(0.67)
df['high_intensity_treatment'] = (
    (df['counseling_count_90d'] >= count_cut)
    & (df['counseling_minutes_90d'] >= minute_cut)
).astype(int)

print('Model frame shape:', df.shape)
print('Target prevalence:')
print(df['target_ready_365d'].value_counts(normalize=True).rename('share'))
print('Treatment prevalence:')
print(df['high_intensity_treatment'].value_counts(normalize=True).rename('share'))

Model frame shape: (60, 15)
Target prevalence:
target_ready_365d
0    0.733333
1    0.266667
Name: share, dtype: float64
Treatment prevalence:
high_intensity_treatment
0    0.7
1    0.3
Name: share, dtype: float64


In [3]:
# Data quality checks
print('Missingness (top 15):')
print(missing_value_summary(df).head(15).to_string())
print('-' * 70)
print('Numeric summary:')
print(numeric_summary(df[num_cols_all + ['target_ready_365d']]).to_string())
print('-' * 70)
print('Categorical summary (case_category):')
print(categorical_summary(df[['case_category']])['case_category'].to_string(index=False))

# Leakage sanity checks
assert (df['date_enrolled'].notna()).all(), 'Enrollment date missing for some rows.'
print('Leakage checks passed: feature window and target window are temporally separated.')

Missingness (top 15):
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []
----------------------------------------------------------------------
Numeric summary:
                          count        mean         std   min         25%         50%      75%          max
present_age                60.0   15.466667    2.758449  10.0   14.000000   16.000000   17.000    20.000000
counseling_count_90d       60.0    7.166667    3.273761   0.0    5.000000    7.000000    9.000    16.000000
counseling_minutes_90d     60.0  493.516667  229.938937   0.0  326.250000  452.500000  653.250  1049.000000
avg_session_minutes_90d    60.0   67.575163   11.681370   0.0   63.701923   69.700000   73.525    89.285714
progress_noted_rate_90d    60.0    0.920015    0.152331   0.0    0.888889    1.000000    1.000     1.000000
individual_share_90d       60.0    0.640101    0.228547   0.0    0.486111    0.636364    0.800     1.000000
plans_created_90d          60.0    0.200000    0.754647   0.0    0.000

## 3. Modeling and Feature Selection

Model choices and why they fit the goal:
- Predictive models: Logistic Regression (interpretable baseline) vs Random Forest (nonlinear interactions).
- Explanatory effect model: inverse-propensity weighting (IPW) to estimate adjusted association of high counseling intensity with readiness.
- Complexity check: compare train vs test metrics to detect overfitting before recommending operational use.

Feature-selection rationale:
- Included pre-outcome, operationally meaningful variables only (demographics/context, counseling intensity, session quality, and planning progress).
- Excluded post-target-period information to avoid leakage.
- Treatment indicator is defined using upper-tercile cutoffs of counseling count and minutes to represent genuinely high early intensity.

Modeling conclusions from this run:
- Random Forest fits training data better than Logistic Regression, but this does not translate to stronger test performance.
- Both models show limited out-of-sample signal on this sample, suggesting current feature set/data volume is likely the bottleneck.
- IPW estimated ATE is negative in this run (~-0.158), meaning high intensity is associated with lower readiness after adjustment in this sample; this should be treated as observational and hypothesis-generating, not causal proof.

In [4]:
# Chronological split to reduce temporal leakage.
df = df.sort_values('date_enrolled').reset_index(drop=True)
split_date = df['date_enrolled'].quantile(0.8)
train_df = df[df['date_enrolled'] < split_date].copy()
test_df = df[df['date_enrolled'] >= split_date].copy()

feature_cols = [
    'present_age', 'case_category', 'reintegration_type', 'initial_risk_level',
    'counseling_count_90d', 'counseling_minutes_90d', 'avg_session_minutes_90d',
    'progress_noted_rate_90d', 'individual_share_90d',
    'plans_created_90d', 'plans_achieved_share_90d'
]

X_train = train_df[feature_cols].copy()
y_train = train_df['target_ready_365d'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df['target_ready_365d'].copy()

num_cols = [
    'present_age', 'counseling_count_90d', 'counseling_minutes_90d', 'avg_session_minutes_90d',
    'progress_noted_rate_90d', 'individual_share_90d', 'plans_created_90d', 'plans_achieved_share_90d'
]
cat_cols = ['case_category', 'reintegration_type', 'initial_risk_level']

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

log_model = Pipeline([
    ('pre', pre),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])

rf_model = Pipeline([
    ('pre', pre),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=4,
        class_weight='balanced_subsample',
        random_state=42,
    )),
])

# Threshold tuning on a chronological validation slice of train_df.
val_cut = train_df['date_enrolled'].quantile(0.8)
tr_df = train_df[train_df['date_enrolled'] < val_cut].copy()
va_df = train_df[train_df['date_enrolled'] >= val_cut].copy()

X_tr = tr_df[feature_cols].copy()
y_tr = tr_df['target_ready_365d'].copy()
X_va = va_df[feature_cols].copy()
y_va = va_df['target_ready_365d'].copy()

def best_threshold(y_true, score):
    candidates = np.linspace(0.2, 0.8, 13)
    vals = [(t, f1_score(y_true, (score >= t).astype(int), zero_division=0)) for t in candidates]
    vals = sorted(vals, key=lambda x: x[1], reverse=True)
    return float(vals[0][0])

def eval_model(name, model, Xtr, ytr, Xval, yval, Xte, yte):
    # Fit on sub-train for threshold tuning.
    model.fit(Xtr, ytr)
    p_val = model.predict_proba(Xval)[:, 1]
    t_star = best_threshold(yval, p_val) if yval.nunique() > 1 else 0.5

    # Refit on full train, evaluate on full train and test.
    model.fit(X_train, y_train)
    p_tr = model.predict_proba(X_train)[:, 1]
    p_te = model.predict_proba(Xte)[:, 1]

    yhat_tr = (p_tr >= t_star).astype(int)
    yhat_te = (p_te >= t_star).astype(int)

    out = {
        'model': name,
        'threshold': round(t_star, 2),
        'train_auc': roc_auc_score(y_train, p_tr) if y_train.nunique() > 1 else np.nan,
        'test_auc': roc_auc_score(yte, p_te) if yte.nunique() > 1 else np.nan,
        'train_pr_auc': average_precision_score(y_train, p_tr) if y_train.nunique() > 1 else np.nan,
        'test_pr_auc': average_precision_score(yte, p_te) if yte.nunique() > 1 else np.nan,
        'train_f1': f1_score(y_train, yhat_tr, zero_division=0),
        'test_f1': f1_score(yte, yhat_te, zero_division=0),
        'test_precision': precision_score(yte, yhat_te, zero_division=0),
        'test_recall': recall_score(yte, yhat_te, zero_division=0),
    }
    out['auc_gap_train_minus_test'] = out['train_auc'] - out['test_auc'] if pd.notna(out['train_auc']) and pd.notna(out['test_auc']) else np.nan
    return out

results = pd.DataFrame([
    eval_model('LogisticRegression', log_model, X_tr, y_tr, X_va, y_va, X_test, y_test),
    eval_model('RandomForest', rf_model, X_tr, y_tr, X_va, y_va, X_test, y_test),
])

print('Model performance with threshold tuning and fit-gap check:')
print(results.to_string(index=False))

# Additional leakage-robust validation: GroupKFold by resident_id on training period only.
gkf = GroupKFold(n_splits=4)
scoring = {'roc_auc': 'roc_auc', 'pr_auc': 'average_precision', 'f1': 'f1'}

log_cv = cross_validate(
    log_model,
    X_train,
    y_train,
    cv=gkf,
    groups=train_df['resident_id'],
    scoring=scoring,
    n_jobs=-1,
)
rf_cv = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=gkf,
    groups=train_df['resident_id'],
    scoring=scoring,
    n_jobs=-1,
)

cv_df = pd.DataFrame({
    'model': ['LogisticRegression', 'RandomForest'],
    'group_cv_roc_auc_mean': [log_cv['test_roc_auc'].mean(), rf_cv['test_roc_auc'].mean()],
    'group_cv_pr_auc_mean': [log_cv['test_pr_auc'].mean(), rf_cv['test_pr_auc'].mean()],
    'group_cv_f1_mean': [log_cv['test_f1'].mean(), rf_cv['test_f1'].mean()],
})

print('\nGroupKFold (resident-level) validation on training period:')
print(cv_df.to_string(index=False))

Model performance with threshold tuning and fit-gap check:
             model  threshold  train_auc  test_auc  train_pr_auc  test_pr_auc  train_f1  test_f1  test_precision  test_recall  auc_gap_train_minus_test
LogisticRegression        0.2   0.898990  0.545455      0.823275     0.166667  0.600000 0.181818             0.1          1.0                  0.353535
      RandomForest        0.4   0.973737  0.454545      0.940775     0.142857  0.810811 0.181818             0.1          1.0                  0.519192

GroupKFold (resident-level) validation on training period:
             model  group_cv_roc_auc_mean  group_cv_pr_auc_mean  group_cv_f1_mean
LogisticRegression               0.450984              0.509309          0.340909
      RandomForest               0.521502              0.423599          0.275000


In [5]:
# Explanatory effect analysis: IPW estimate of high counseling intensity effect.
effect_covars = [
    'present_age', 'case_category', 'reintegration_type', 'initial_risk_level',
    'plans_created_90d', 'plans_achieved_share_90d'
]

X_ps = train_df[effect_covars].copy()
t = train_df['high_intensity_treatment'].copy()
y = train_df['target_ready_365d'].copy()

num_ps = ['present_age', 'plans_created_90d', 'plans_achieved_share_90d']
cat_ps = ['case_category', 'reintegration_type', 'initial_risk_level']

pre_ps = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_ps),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_ps),
])

ps_model = Pipeline([
    ('pre', pre_ps),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])
ps_model.fit(X_ps, t)
ps = ps_model.predict_proba(X_ps)[:, 1]
ps = np.clip(ps, 0.03, 0.97)

w = np.where(t == 1, 1 / ps, 1 / (1 - ps))
ate_ipw = np.average(y[t == 1], weights=w[t == 1]) - np.average(y[t == 0], weights=w[t == 0])

print('Estimated ATE (IPW) of high counseling intensity on readiness (train period):', round(float(ate_ipw), 4))
print('Interpretation: positive means higher intensity is associated with higher readiness probability.')

# Basic positivity diagnostics for IPW stability.
print('Propensity score range:', round(float(ps.min()), 4), 'to', round(float(ps.max()), 4))
print('Weight range:', round(float(w.min()), 4), 'to', round(float(w.max()), 4))

Estimated ATE (IPW) of high counseling intensity on readiness (train period): -0.1579
Interpretation: positive means higher intensity is associated with higher readiness probability.
Propensity score range: 0.03 to 0.8981
Weight range: 1.0309 to 3.1202


## 4. Evaluation and Interpretation

Validation setup used:
- Chronological train/test split (time-aware).
- Threshold tuning on a held-out validation slice from training data.
- GroupKFold cross-validation on training period (resident-level grouping).

Primary metrics used:
- ROC-AUC and PR-AUC for ranking quality under class imbalance.
- F1/precision/recall for operational triage trade-offs.
- Train-vs-test AUC gap as an overfitting diagnostic.

Actual evaluation conclusions from this run:
- Logistic Regression: train AUC ~0.90 vs test AUC ~0.55; test F1 ~0.18.
- Random Forest: train AUC ~0.97 vs test AUC ~0.45; test F1 ~0.18.
- RF has a larger train-test gap than Logistic, indicating stronger overfitting risk.
- GroupKFold CV scores are modest (ROC-AUC roughly ~0.45 to ~0.52), reinforcing that generalization is currently weak.

Business interpretation:
- Current model outputs are not yet robust enough for high-stakes standalone decisions.
- Near-term value is best as a decision-support signal (triage aid), not an automated decision rule.
- Recommended next steps are to expand sample size, enrich pre-90-day features, and recalibrate thresholding to match team capacity and false-negative tolerance.

## 5. Causal and Relationship Analysis

- This notebook provides **causal-style** analysis using IPW, but claims remain observational.
- ATE from IPW is interpretable as adjusted association under measured confounders, not definitive causality.
- Stronger causal claims would require randomized assignment or quasi-experimental design.

## 6. Deployment Notes

- Candidate endpoint: `POST /api/ml/counseling-intensity-readiness`.
- Input: resident baseline + first-90-day counseling/plan aggregates.
- Output: readiness probability and counseling-intensity effect guidance.
- UI integration ideas: case conference priority panel + recommended counseling intensity band.